# Exploratory Data Analysis (EDA)

There are four data sources in `.csv` format (see README for more details):
- MIT-BIH Supraventricular Arrhythmia Database
- MIT-BIH Arrhythmia Database
- St. Petersburg INCART 12-lead Arrhythmia Database
- Sudden Cardiac Death Holter Database

This is a supervised machine learning project using a neural classifier model. Our `target` has the following classes:
- N = Normal
- SVEB = SupraVentricular Ectopic Beat
- VEB = Ventricular Ectopic Beat
- F = Fusion beat 
- Q = Unclassifiable

SVEB, VEB, F, and Q are considered "abnormal" classes, where N is considered as "normal". The goal of the model is therefore to predict a `target` class as either "abnormal" or "normal". A confidence score will also be given alongside the prediction. 

There are a total of 32 `features` (16 per lead).

## Goals of this notebook
> *note: these could be scoped in to other notebooks as needed*

### _raw data_
- shape?
- column names & dtypes?
- null values/empty strings?
- can I join all four datasets safely?
### _target variable_
- what is the class distribution of the 'type' column?
- is class imbalance 88/12 (normal/arrhythmia) as annotated? confirm this
- what is the class distribution after binary mapping? 
- do I drop Q? how much data would I lose? 
### _features_
- similar ranges?
- any low or near-zero-variance?
### _resampling_
- run SMOTE
- class distribution before resampling? after?
### _train/test split_
- are target class proportions preserved when we split the data?
### _data quality_
- any outliers in RR intervals or peak vals?
- anything else that might break pipeline? 

--- 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# load all data, adding a source column for eventual concatenation
incart = pd.read_csv('../data/incart-2lead-arrhythmia.csv', dtype={'type': str}).assign(source='incart')
mb_arr = pd.read_csv('../data/mit-bih-arrhythmia.csv', dtype={'type': str}).assign(source='mb_arr')
mb_supra = pd.read_csv('../data/mit-bih-supraventricular.csv', dtype={'type': str}).assign(source='mb_supra')
holter = pd.read_csv('../data/sudden-cardiac-death-holter.csv', dtype={'type': str}).assign(source='holter')

In [ ]:
holter.dtypes

In [ ]:
# shapes
data = [incart, mb_arr, mb_supra, holter]
for df in data:
    print(df.shape)


In [ ]:
# check if features are named the same
ref = set(incart.columns)
others = [mb_arr, mb_supra, holter]
for df in others:
    print(set(df.columns) == ref)

In [ ]:
# null values
for df in data:
    print(df.isnull().sum())

In [ ]:
# check it is the holter one giving us nulls
holter['type'].isnull().sum()

In [ ]:
# check for empty strings
(holter == '').sum()

In [ ]:
# nulls in 'type' col
holter[holter['type'].isnull()]


In [ ]:
# check for nulls on the second lead
holter[
    holter['record'].notnull() & holter['1_pre-RR'].isnull()
]

In [ ]:
# list indeces of nulls
null_positions = holter[holter['record'].isnull()].index
null_positions


In [ ]:
# how many nulls
null_positions.max() - null_positions.min() + 1 == len(null_positions)

In [ ]:
np.diff(null_positions)

In [ ]:
(np.diff(null_positions) != 1).sum()

In [ ]:
null_positions.min()

In [ ]:
null_positions.max()

In [ ]:
len(holter)

In [ ]:
# visualise nulls by index: value of 1 is a null row
# call this group A: where the identifiers and 0-lead are missing
plt.plot(holter['record'].isnull())
plt.title('NULL GROUP A: no identifiers and 0_lead')
plt.show()

In [ ]:
# repeat for the other case, group B
# less severe, as identifiers ('record', 'type') are intact
plt.plot(
    holter['record'].notnull() & holter['1_pre-RR'].isnull()
)
plt.title('NULL GROUP B: no 1_lead')
plt.show

In [ ]:
len(incart) + len(mb_arr) + len(mb_supra)

In [ ]:
# drop all rows with null values (groups A & B)
holter = holter.dropna()

In [ ]:
len(holter)

In [ ]:
holter.info()

In [ ]:
for df in [holter, mb_arr, mb_supra]:
    df['record'] = df['record'].astype(str)

In [ ]:
for df in data:
    numeric_cols = df.select_dtypes(include='number').columns
    df[numeric_cols] = df[numeric_cols].astype('float64')

In [ ]:
# build a subset and look at all dtypes for the df
dtype_compare = pd.DataFrame({
    'incart': incart.dtypes,
    'mb_arr': mb_arr.dtypes,
    'mb_supra': mb_supra.dtypes,
    'holter': holter.dtypes
})

dtype_compare

In [ ]:
all_data = pd.concat([incart, mb_arr, mb_supra, holter], ignore_index=True)
all_data

In [ ]:
len(incart) + len(mb_arr) + len(mb_supra) + len(holter)

In [ ]:
all_data.info()

In [ ]:
all_data['record'].nunique()

In [ ]:
# check to see if number of unique records == the composite key we will use for train/test split
all_data[['source','record']].drop_duplicates().shape[0] == all_data['record'].nunique()

In [ ]:
all_data['type'].value_counts(normalize=True)